# Этап 4. Признаки персонального ранжирования
Запускайте ячейки сверху вниз из корневой папки проекта после этапа 3. Нужны Python, pandas и NumPy. Работа ведётся с локальным архивом. Каждый период обрабатывается отдельно. Параметр сглаживания a = 10 задан заранее: общая средняя имеет вес десяти оценок. Группы и метки формируются для ретроспективного эксперимента. При построении признаков используются только прошлые оценки и жанровый справочник.

In [ ]:
from pathlib import Path
import hashlib
import json
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
archive = root / "data" / "ml-100k.zip"
previous = root / "results_stage3"
out = root / "results_stage4"
out.mkdir(exist_ok=True)
metadata = json.loads((previous / "versions.json").read_text(encoding="utf-8"))
if hashlib.sha256(archive.read_bytes()).hexdigest() != metadata["sha256"]:
    raise ValueError("Архив отличается от использованного на этапе 3")
periods = pd.read_csv(previous / "periods.csv").set_index("период")
expected = pd.read_csv(previous / "eligibility.csv").set_index("период")
with zipfile.ZipFile(archive) as bundle:
    def read_file(name, sep):
        with bundle.open("ml-100k/" + name) as stream:
            return pd.read_csv(stream, sep=sep, header=None, encoding="latin-1",
                               keep_default_na=False)
    events = read_file("u.data", "\t")
    genres = read_file("u.genre", "|").sort_values(1)[0].tolist()
    movies = read_file("u.item", "|").set_index(0)
events.columns = ["user", "item", "rating", "timestamp"]
events["time"] = pd.to_datetime(events.timestamp, unit="s", utc=True)
genre = movies.iloc[:, 4:].copy()
genre.columns = ["genre_" + name for name in genres]
genre = genre.astype("uint8")
if not genre.isin([0, 1]).all().all():
    raise ValueError("Ошибка в жанровой разметке")
a = 10
print("Каталог результатов:", out)

## Группы и смысл полей
Файл feature_groups.json задаёт списки входных столбцов для последующего обучения и исключения групп. user, item и label служат для группировки, проверки и оценки.

In [ ]:
groups = {
    "user": ["user_count", "user_mean", "user_positive_share"],
    "item": ["item_count", "item_positive_count", "item_smoothed_mean", *genre.columns],
    "pair": ["genre_cosine", "positive_overlap_share", "history_overlap_count"]
}
features = [col for cols in groups.values() for col in cols]
# Два фильма пересекаются, если есть хотя бы один общий флаг, включая unknown.
# Матрица использует только постоянные характеристики фильмов.
genre_values = genre.to_numpy()
overlap = pd.DataFrame((genre_values @ genre_values.T) > 0,
                       index=genre.index, columns=genre.index)

## Расчёт признаков на одну дату
Сначала строится каталог известных фильмов и исторические агрегаты. Затем по каждой истории исключаются оценённые фильмы. Функция ниже не получает ни будущих оценок, ни целевых меток. Для пустого положительного профиля сходство и доля пересечений равны нулю. На этапе моделирования жанровый алгоритм сможет распознать этот случай по user_positive_share и применить популярность.

In [ ]:
def item_features(history):
    scores = history.assign(positive=history.rating.ge(4)).groupby("item").agg(
        item_count=("rating", "size"), item_positive_count=("positive", "sum"),
        rating_sum=("rating", "sum"))
    scores["item_smoothed_mean"] = ((scores.pop("rating_sum") + a * history.rating.mean())
                                    / (scores.item_count + a))
    return scores.join(genre)

def user_candidates(past, catalog):
    user_id = past.user.iloc[0]
    candidates = catalog.index[~catalog.index.isin(past.item)]
    rows = catalog.loc[candidates].copy()
    liked = past.loc[past.rating.ge(4), "item"]
    rows["user"] = user_id
    rows["user_count"] = len(past)
    rows["user_mean"] = past.rating.mean()
    rows["user_positive_share"] = len(liked) / len(past)
    rows["history_overlap_count"] = overlap.loc[candidates, past.item].sum(axis=1)
    rows["genre_cosine"] = 0.0
    rows["positive_overlap_share"] = 0.0
    if len(liked):
        profile = genre.loc[liked].mean(axis=0).to_numpy()
        vectors = genre.loc[candidates].to_numpy()
        denominators = np.linalg.norm(vectors, axis=1) * np.linalg.norm(profile)
        rows["genre_cosine"] = np.divide(vectors @ profile, denominators,
                                         out=np.zeros(len(rows)), where=denominators > 0)
        rows["positive_overlap_share"] = overlap.loc[candidates, liked].mean(axis=1)
    return rows.reset_index()

## Разметка, проверки и сохранение
Положительные события периода используются для разметки и отбора групп по правилам этапа 3. Для каждого допущенного пользователя сохраняется полный список кандидатов. Файлы train, validation и test используют один порядок столбцов. Группа при объединении периодов определяется парой «период, пользователь».

In [ ]:
summary = []
examples = []
for name, stem in [("Обучение", "train"), ("Валидация", "validation"), ("Тест", "test")]:
    start = pd.Timestamp(periods.loc[name, "начало UTC"])
    end = pd.Timestamp(periods.loc[name, "конец UTC исключительно"])
    history = events.loc[events.time < start]
    future = events.loc[(events.time >= start) & (events.time < end)]
    catalog = item_features(history)
    targets = future.loc[future.rating.ge(4)].groupby("user").item.agg(set).to_dict()
    blocks = []
    for user_id, past in history.groupby("user", sort=True):
        relevant = targets.get(user_id, set()).intersection(catalog.index).difference(past.item)
        if not relevant or len(catalog) - past.item.nunique() < 10:
            continue
        rows = user_candidates(past, catalog)
        rows["label"] = rows.item.isin(relevant).astype("uint8")
        blocks.append(rows)
    if not blocks:
        raise ValueError(f"{name}: нет допустимых групп")
    data = pd.concat(blocks, ignore_index=True)[["user", "item", "label", *features]]
    sizes = data.groupby("user").label.agg(["size", "sum"])
    mixed = ((sizes["sum"] > 0) & (sizes["sum"] < sizes["size"])).sum()
    counts = [len(catalog), len(sizes), int(mixed), int(data.label.sum())]
    reference = expected.loc[name, ["каталог", "допущены к оценке",
                                    "допущены с обеими метками",
                                    "из них у допущенных пользователей"]].tolist()
    if counts != reference:
        raise ValueError(f"{name}: получено {counts}, ожидалось {reference}")
    if data.duplicated(["user", "item"]).any() or not np.isfinite(data[features]).all().all():
        raise ValueError(f"{name}: повторные пары или неконечные значения признаков")
    if sizes["size"].min() < 10 or not data.genre_cosine.between(0, 1 + 1e-10).all():
        raise ValueError(f"{name}: проверьте размеры групп и сходство")
    data.to_csv(out / f"{stem}.csv.gz", index=False, compression="gzip")
    summary.append([name, start, len(history), len(catalog), len(sizes), len(data),
                    int(data.label.sum()), int(mixed), int(sizes["size"].min()),
                    int(sizes["size"].max())])
    if stem == "train":
        first_user = data.user.min()
        seen = history.loc[history.user.eq(first_user)]
        liked = seen.loc[seen.rating.ge(4)]
        sample = data.loc[data.user.eq(first_user)].drop_duplicates(subset=genre.columns.tolist()).head(2)
        for _, row in sample.iterrows():
            item = int(row["item"])
            item_genres = genre.loc[item].astype(bool)
            matches = genre.loc[seen.item, item_genres].any(axis=1).sum()
            positive_matches = genre.loc[liked.item, item_genres].any(axis=1).sum()
            share = positive_matches / len(liked) if len(liked) else 0.0
            if matches != row.history_overlap_count or not np.isclose(share, row.positive_overlap_share):
                raise ValueError("Контрольный подсчёт пересечений не совпал")
            examples.append([first_user, item, movies.loc[item, 1],
                             ", ".join(np.array(genres)[item_genres.to_numpy()]),
                             len(seen), len(liked), matches, positive_matches, share,
                             row.genre_cosine])
    print(name, "— сверка пройдена")

In [ ]:
report = pd.DataFrame(summary, columns=["период", "граница UTC", "оценки в истории",
    "фильмы в каталоге", "группы", "кандидатные пары", "положительные пары",
    "группы с обеими метками", "минимум кандидатов", "максимум кандидатов"])
example = pd.DataFrame(examples, columns=["пользователь", "фильм", "название", "жанры",
    "прошлые оценки", "прошлые положительные", "оценённые с общим жанром",
    "положительные с общим жанром", "доля пересечения", "косинусное сходство"])
report.to_csv(out / "feature_summary.csv", index=False, encoding="utf-8-sig")
example.to_csv(out / "feature_example.csv", index=False, encoding="utf-8-sig")
(out / "feature_groups.json").write_text(json.dumps(groups, indent=2), encoding="utf-8")
display(report, example)